# FIFA 21 Data Cleaning & Exploratory Data Analysis

This project focuses on cleaning and transforming a messy scraped FIFA 21 dataset into an analysis-ready format using Python and Jupyter Notebook.

The dataset required extensive preprocessing due to inconsistent formatting in financial values, player attributes, and contract details.

In [ ]:
import pandas as pd
import numpy as np



In [ ]:
df = pd.read_csv("fifa21_raw_data.csv")

## 1. Data Overview

The dataset was scraped from sofifa.com and contains player information including height, weight, value, wage, contract details, and performance ratings.

Due to web scraping inconsistencies, several columns required cleaning and type conversion.

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

## 2. Data Cleaning

The following transformations were performed:

- Converted height from feet/inches to centimeters
- Converted weight to numeric format
- Converted Value, Wage, and Release Clause into numeric euro values
- Removed star characters from rating columns
- Removed newline characters

##### Clean Height and Weight #

In [ ]:
df["Height"].unique()[:20]

In [ ]:
import numpy as np

def convert_height(height):
    try:
        height = str(height).strip()
        
        if "'" in height:
            parts = height.split("'")
            feet = int(parts[0])
            inches = int(parts[1].replace('"', ''))
            return round((feet * 30.48) + (inches * 2.54), 2)
        
        return np.nan
    
    except:
        return np.nan

df["Height_cm"] = df["Height"].apply(convert_height)

In [ ]:
import numpy as np

def convert_height(height):
    try:
        height = str(height).strip()
        
        if "'" in height:
            parts = height.split("'")
            feet = int(parts[0])
            inches = int(parts[1].replace('"', ''))
            return round((feet * 30.48) + (inches * 2.54), 2)
        
        return np.nan
    
    except:
        return np.nan

df["Height_cm"] = df["Height"].apply(convert_height)

In [ ]:
df[["Height", "Height_cm"]].head(10)

In [ ]:
df["Height_cm"].describe() #155 cm → shorter players

#206 cm → tall goalkeepers

#### Convert Weight to Kilograms #

In [ ]:
df["Weight_kg"] = (
    df["Weight"]
    .str.replace("lbs", "", regex=False)
    .astype(float) * 0.453592
)

df[["Weight", "Weight_kg"]].head()

In [ ]:
df["Weight_kg"].describe() #to validate the change
#49.8 kg → lightweight winger or young player ✅

#110 kg → big defender/goalkeeper

#### Value, Wage & Release Clause #

In [ ]:
df[["Value", "Wage", "Release Clause"]].head(10)

In [ ]:
import numpy as np

def convert_currency(value):
    try:
        value = str(value).strip()
        value = value.replace('€', '')
        
        if 'M' in value:
            return float(value.replace('M', '')) * 1_000_000
        
        elif 'K' in value:
            return float(value.replace('K', '')) * 1_000
        
        else:
            return float(value)
    
    except:
        return np.nan

In [ ]:
df["Value_num"] = df["Value"].apply(convert_currency)
df["Wage_num"] = df["Wage"].apply(convert_currency)
df["ReleaseClause_num"] = df["Release Clause"].apply(convert_currency)

In [ ]:
df[["Value", "Value_num"]].head()
df["Value_num"].describe() #Min = 0 → players with zero market value (probably free agents, low-rated players, or missing data). Totally normal.

#Max = 105,500,000 → that’s €105.5M. Looks like FIFA 21’s top stars. ✅

In [ ]:
df[["Name", "Value", "Value_num"]].sort_values(by="Value_num", ascending=False).head(10) #You’ll see the top 10 most valuable players. That’s your sanity check.

In [ ]:
df.isnull().sum()
df.duplicated().sum() #Ensure all columns you want to showcase are clean and consistent

# 3. Exploratory Data Analysis: now we tell the story using visuals
This section explores the relationship between player market value and wages to identify potential underpaid but highly valuable players.

## Underpaid players

In [ ]:
#Scatter Plot: Underpaid Stars
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.scatter(df["Wage_num"], df["Value_num"], alpha=0.5, color='green')
plt.xlabel("Wage (€)")
plt.ylabel("Value (€)")
plt.title("FIFA 21: Underpaid but Valuable Players")
plt.grid(True)
# Save plot as image
plt.savefig("FIFA21_value_vs_wage.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
df.columns

## long-term players

In [ ]:
import pandas as pd
import numpy as np

# --- Make sure 'Joined' is datetime ---
df["Joined"] = pd.to_datetime(df["Joined"], errors='coerce')

# --- Create 'Years_at_club' safely ---
df["Years_at_club"] = np.where(df["Joined"].notna(), 2021 - df["Joined"].dt.year, np.nan)

# --- Filter players who have been at a club for more than 10 years ---
long_term_players = df[df["Years_at_club"] > 10]

# --- Select relevant columns & sort descending ---
long_term_players = long_term_players[["Name", "Team & Contract", "Years_at_club"]].sort_values(by="Years_at_club", ascending=False)

# --- Show top 10 longest-serving players ---
long_term_players.head(10)

In [ ]:
# Example (if not already done)
df["Value_num"] = df["Value"].apply(convert_currency)
df["Wage_num"] = df["Wage"].apply(convert_currency)

# 4. Feature Engineering

A new feature `Years_at_club` was created using the 'Joined' column to determine player loyalty and long-term tenure.

In [ ]:
def convert_currency(value):
    try:
        value = str(value).strip().replace('€','')
        if 'M' in value:
            return float(value.replace('M','')) * 1_000_000
        elif 'K' in value:
            return float(value.replace('K','')) * 1_000
        else:
            return float(value)
    except:
        return np.nan

df["Wage_num"] = df["Wage"].apply(convert_currency)
df["Value_num"] = df["Value"].apply(convert_currency)

In [ ]:
# Step 1: Convert Wage and Value to numeric in the original df
def convert_currency(value):
    try:
        value = str(value).strip().replace('€','')
        if 'M' in value:
            return float(value.replace('M','')) * 1_000_000
        elif 'K' in value:
            return float(value.replace('K','')) * 1_000
        else:
            return float(value)
    except:
        return np.nan

df["Wage_num"] = df["Wage"].apply(convert_currency)
df["Value_num"] = df["Value"].apply(convert_currency)

# Step 2: Filter long-term players AND make a copy
long_term_players = df[df["Years_at_club"] > 10].copy()

In [ ]:
long_term_players[["Name", "Team & Contract", "Years_at_club", "Wage_num", "Value_num"]].head(10)

## Scattered plot of underpaid vs high valued players

In [ ]:
#
import matplotlib.pyplot as plt

plt.figure(figsize=(12,8))

plt.scatter(df["Wage_num"], df["Value_num"], alpha=0.3, label="Other Players")
plt.scatter(
    long_term_players["Wage_num"], 
    long_term_players["Value_num"], 
    color='red', 
    alpha=0.7, 
    label="Players >10 years at club"
)

plt.xlabel("Wage (€)")
plt.ylabel("Market Value (€)")
plt.title("FIFA 21: Underpaid vs High-Value Players")
plt.legend()
plt.grid(True)
plt.show()

## underpaid but valuable players

In [ ]:
# Filter underpaid but valuable players
underpaid_gems = df[(df["Value_num"] > 20_000_000) & (df["Wage_num"] < 10_000)]

# Select the columns we want to show
underpaid_gems = underpaid_gems[["Name", "Team & Contract", "Value_num", "Wage_num", "Years_at_club"]]

# Sort by Value descending
underpaid_gems = underpaid_gems.sort_values(by="Value_num", ascending=False)

# Show top 10 gems
underpaid_gems.head(10)

In [ ]:
##Red dots → loyal veterans

#Gold dots → underpaid gems

#Scatter clearly shows high-value but low-wage players, perfect for storytelling.
plt.figure(figsize=(12,8))

# All players
plt.scatter(df["Wage_num"], df["Value_num"], alpha=0.3, label="Other Players")

# Long-term loyal players (>10 yrs)
plt.scatter(
    long_term_players["Wage_num"], 
    long_term_players["Value_num"], 
    color='red', 
    alpha=0.7, 
    label="Players >10 yrs at club"
)

# Underpaid gems
plt.scatter(
    underpaid_gems["Wage_num"], 
    underpaid_gems["Value_num"], 
    color='gold', 
    edgecolors='black', 
    s=100, 
    label="Underpaid Gems"
)

plt.xlabel("Wage (€)")
plt.ylabel("Market Value (€)")
plt.title("FIFA 21: Underpaid vs High-Value Players")
plt.legend()
plt.grid(True)
plt.show()

# 5. Key Insights

1. Player market values ranged up to €105.5M after cleaning and conversion.
2. Several players have remained at their clubs for over 10 years, indicating strong loyalty.
3. A subset of players demonstrate high market value while earning relatively low wages, suggesting potential undervaluation.
4. The dataset required significant preprocessing due to inconsistent string formatting from web scraping.

# 6. Conclusion

This project demonstrates practical data cleaning, feature engineering, and exploratory data analysis skills using real-world messy data.

The workflow ensures reproducibility and structured transformation from raw scraped data to analysis-ready insights.

In [ ]:
plt.savefig("FIFA21_value_vs_wage.png")

In [ ]:
import os

# Create a folder called 'plots' if it doesn't exist
os.makedirs("plots", exist_ok=True)

# Check where it's created
print("Current working directory:", os.getcwd())
print("Plots folder ready at:", os.path.join(os.getcwd(), "plots"))

In [ ]:
plt.savefig("plots/FIFA21_value_vs_wage.png", dpi=300, bbox_inches='tight')

In [ ]:
print(os.listdir("plots"))

In [ ]:
import os
import matplotlib.pyplot as plt

# ------------------------------
# Step 1: Create plots folder
# ------------------------------
os.makedirs("plots", exist_ok=True)
print(f"Plots folder ready at: {os.path.join(os.getcwd(), 'plots')}")

# ------------------------------
# Step 2: Value vs Wage Scatter Plot
# ------------------------------
plt.figure(figsize=(12,8))
plt.scatter(df["Wage_num"], df["Value_num"], alpha=0.3, label="Other Players")
plt.scatter(
    long_term_players["Wage_num"], 
    long_term_players["Value_num"], 
    color='red', alpha=0.7, label="Players >10 yrs"
)
plt.scatter(
    underpaid_gems["Wage_num"], 
    underpaid_gems["Value_num"], 
    color='gold', edgecolors='black', s=100, label="Underpaid Gems"
)
plt.xlabel("Wage (€)")
plt.ylabel("Market Value (€)")
plt.title("FIFA 21: Underpaid vs High-Value Players")
plt.legend()
plt.grid(True)

# Save plot
plt.savefig("plots/FIFA21_value_vs_wage.png", dpi=300, bbox_inches='tight')
plt.show()

# ------------------------------
# Step 3: Height vs Weight Scatter Plot (optional)
# ------------------------------
plt.figure(figsize=(10,6))
plt.scatter(df["Height_cm"], df["Weight_kg"], alpha=0.5, c='blue')
plt.xlabel("Height (cm)")
plt.ylabel("Weight (kg)")
plt.title("FIFA 21: Height vs Weight Distribution")
plt.grid(True)

# Save plot
plt.savefig("plots/FIFA21_height_vs_weight.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os
print(os.getcwd())  # Shows where your notebook is running from